# EDA e Baseline — Previsão de Churn (Telco)

**Autor:** Alessandro Stefani  
**Data:** 2026-06-21  
**Disciplina:** Tech Challenge — Fase 01 (Produtização de Modelos)  
**Etapa:** 1 — Entendimento e Preparação  
**Descrição:** Análise exploratória do dataset Telco Customer Churn e construção de baselines (DummyClassifier e Regressão Logística), com rastreamento no MLflow.

**Dataset:** TelcoCustomerChurn.csv  
**Disponivel em:** https://www.kaggle.com/datasets/rhonarosecortez/telco-customer-churn/data

## 1. Setup e imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer

import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score


dataset_path = "../data/raw/TelcoCustomerChurn.csv"

SEED = 42

## 2. Carregamento dos dados

Dataset bruto em `data/raw/telco.csv`.

In [2]:
# Carregar dados
df = pd.read_csv(dataset_path)

print(f"Shape dos dados: {df.shape}")
print("\nPrimeiras linhas do dataset:")
df.head(10)

Shape dos dados: (7043, 50)

Primeiras linhas do dataset:


,CustomerID,Gender,Age,Under30,SeniorCitizen,Married,Dependents,NumberofDependents,Country,State,...,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,SatisfactionScore,CustomerStatus,ChurnLabel,ChurnScore,CLTV,ChurnCategory,ChurnReason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges
5,4412-YLTKF,Female,72,No,Yes,No,Yes,1,United States,California,...,10,89.91,2235.41,1,Churned,Yes,95,4638,Competitor,Competitor had better devices
6,0390-DCFDQ,Female,76,No,Yes,Yes,Yes,2,United States,California,...,0,15.28,85.73,2,Churned,Yes,76,3964,Other,Don't know
7,3445-HXXGF,Male,66,No,Yes,Yes,No,0,United States,California,...,0,0.00,2610.25,1,Churned,Yes,91,5444,Dissatisfaction,Service dissatisfaction
8,2656-FMOKZ,Female,70,No,Yes,No,Yes,2,United States,California,...,0,661.05,1806.75,2,Churned,Yes,91,5717,Dissatisfaction,Limited range of services
9,2070-FNEXE,Female,77,No,Yes,No,Yes,2,United States,California,...,0,188.65,681.20,2,Churned,Yes,81,4419,Price,Lack of affordable download/upload speed


## 3. Análise Exploratória (EDA)

### 3.1 Visão geral e volume

Dimensões, primeiras linhas, tipos de cada coluna.

In [ ]:
# Visão geral do dataset

print("Formato:")
print(f"Linhas  {df.shape[0]:,}")
print(f"Colunas {df.shape[1]:,}")

print("\n" + "Tipos de colunas:")
print(df.dtypes.value_counts())


### 3.2 Qualidade dos dados

Valores ausentes, duplicados, tipos inconsistentes, cardinalidade das categóricas.

In [ ]:
# Verificação de nulos

nulls = (
    pd.DataFrame({
        "coluna": df.columns,
        "nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values
    })
    .sort_values("pct_nulos", ascending=False)
)

#print("\n" + "=" * 80)
print("Colunas com nulos")
#print("=" * 80)

display(nulls.query("nulos > 0"))


**ChurnReason:** Não será utilizada, pois contém informação sobre a variável resposta (data leakage)  
**ChurnCategory:**  Não será utilizada, pois contém informação sobre a variável resposta (data leakage)  
**Offer:** De acordo com a descrição do dataset, o cliente não tem campanha ativa, então o nulo carrega informação. Popular com "No offer"  
**InternetType:** Os valores nulos são de clientes que não possuem serviço de internet (confirmado com InternetService). Avaliar qual das duas tem mais valor para o modelo  


In [ ]:
# Verificação de cardinalidade

cardinalidade = (
    pd.DataFrame({
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str),
        "valores_unicos": df.nunique(dropna=False).values,
        "pct_unicos": (
            df.nunique(dropna=False)
            / len(df)
            * 100
        ).round(2).values
    })
    .sort_values("valores_unicos")
)

print("Cardinalidade" + "\n")

display(cardinalidade)


In [ ]:
# Análise estatística das variáveis numéricas

print("Descrição das variáveis numéricas" + "\n")

display(df.describe().round(10).T)


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(numeric_cols)

In [ ]:
# Colunas numéricas (excluindo geográficas e leakage confirmado/suspeito)
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = numeric_cols.drop(['ZipCode', 'Latitude', 'Longitude', 'Population'])
numeric_cols = numeric_cols.drop(['TotalRevenue', 'ChurnScore', 'CLTV', 'SatisfactionScore'])
numeric_cols = numeric_cols.drop(['NumberofDependents', 'Number_of_Referrals', 'TotalRefunds', 'TotalExtraDataCharges'])

n = len(numeric_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[idx], color='coral')
    axes[idx].set_title(f'Boxplot: {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].grid(axis='x', alpha=0.3)
    col_zscore = np.abs(stats.zscore(df[col].dropna()))
    outlier_count = (col_zscore > 3).sum()
    axes[idx].text(0.95, 0.95, f'Outliers: {outlier_count}', 
                   transform=axes[idx].transAxes, fontsize=9,
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(facecolor='white', alpha=0.5, edgecolor='gray'))

for ax in axes[n:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()


In [ ]:
# Colunas numéricas (excluindo geográficas e leakage confirmado/suspeito)
n = len(numeric_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[idx], color='coral')
    axes[idx].set_title(f'Boxplot: {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].grid(axis='x', alpha=0.3)
    col_zscore = np.abs(stats.zscore(df[col].dropna()))
    outlier_count = (col_zscore > 3).sum()
    axes[idx].text(0.95, 0.95, f"skew {df[col].skew():.2f}",
                   transform=axes[idx].transAxes, ha="right", va="top",
                   bbox=dict(facecolor="white", alpha=0.5, edgecolor="gray"))

for ax in axes[n:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()


In [ ]:
# Análise de relacionamento entre ZipCode e Latitude/Longitude

(df.groupby(["Latitude", "Longitude"])["ZipCode"].nunique().describe())

In [ ]:
# Verificação de TotalRevenue ser calculada a partir de outras

revenue_calculada = (
    df["TotalCharges"]
    + df["TotalLongDistanceCharges"]
    + df["TotalExtraDataCharges"]
    - df["TotalRefunds"]
)

diferenca = df["TotalRevenue"] - revenue_calculada

display(diferenca.describe().round(10))

In [ ]:
# Análise de SatisfactionScore ser possível data leakage

pd.crosstab(
    df["TotalRefunds"],
    df["ChurnLabel"],
    margins=True
)


#numeric_cols = numeric_cols.drop(['TotalRevenue', 'ChurnScore', 'CLTV', 'SatisfactionScore'])
#numeric_cols = numeric_cols.drop(['NumberofDependents', 'Number_of_Referrals', 'TotalRefunds', 'TotalExtraDataCharges'])


In [ ]:
# Análise das variáveis categóricas

print("Descrição das variáveis categóricas" + "\n")

display(df.describe(include="object").T)


In [ ]:
# Análise de correlação das variáveis

plt.figure(figsize=(14, 10))

sns.heatmap(
    df.select_dtypes(include="number").corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.show()

### 3.3 Distribuições e relação com o alvo (Churn)

Balanceamento da classe alvo; distribuição das numéricas e categóricas; relação com o churn.

In [ ]:
print(df["ChurnLabel"].value_counts())


(df["ChurnLabel"]
    .value_counts(normalize=True)
    .mul(100)
    .plot(kind="bar")
)

plt.ylabel("%")
plt.title("Distribuição da Target")
plt.show()


dist = (
    df["ChurnLabel"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

print(f"Classe majoritária: {dist.idxmax()} ({dist.max()}%)")
print(f"Classe minoritária: {dist.idxmin()} ({dist.min()}%)")


dist = df["ChurnLabel"].value_counts()

maior = dist.max()
menor = dist.min()

ratio = menor / maior
print(f"\nRatio de balanceamento: {ratio:.2f}")
if ratio < 0.5:
    print("⚠️ Dataset desbalanceado!")
else:
    print("✓ Dataset razoavelmente balanceado.")

In [ ]:
# Análise de correlação de variáveis numéricas com target
# Excluindo variáveis geográficas e com leakage possível/confirmado

cols_fora = ['ZipCode', 'Latitude', 'Longitude', 'Population', 'CLTV', 'TotalRevenue', 'ChurnScore']

churn_num = (
    df["ChurnLabel"]
      .map({
          "No": 0,
          "Yes": 1
      })
)

corr_target = (
    df.select_dtypes(include="number")
      .drop(columns=cols_fora)
      .corrwith(churn_num)
      .sort_values()
)

display(
    corr_target.to_frame(
        "Correlacao_Com_Churn"
    )
)

In [ ]:
# Análise gráfica de relacionamento entre variáveis categóricas e target
# Excluindo variáveis de identificação, geográficas, redundantes e com leakage possível/confirmado

TARGET = "ChurnLabel"

cat_cols = df.select_dtypes(include="object").columns
cat_cols = cat_cols.drop(['CustomerID', 'ChurnReason', 'ChurnCategory', 'CustomerStatus', 'City'])
cat_cols = cat_cols.drop(['Country', 'State', 'Quarter'])
cat_cols = cat_cols.drop(['Under30', 'SeniorCitizen']).tolist()


cat_cols = [col for col in cat_cols if col != TARGET]

for col in cat_cols:
    churn_rate = (
        pd.crosstab(
            df[col],
            df[TARGET],
            dropna=False,
            normalize="index"
        )
        .mul(100)
        .round(2)
    )

    if "Yes" not in churn_rate.columns:
        continue

    churn_rate["Yes"].sort_values().plot(
        kind="barh",
        figsize=(8, 4)
    )

    plt.title(f"Taxa de Churn por {col}")
    plt.xlabel("% Churn")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
# Análise numérica de relacionamento entre variáveis categóricas e target
# Excluindo variáveis de identificação, geográficas, redundantes e com leakage possível/confirmado

TARGET = "ChurnLabel"

cat_cols = df.select_dtypes(include="object").columns
cat_cols = cat_cols.drop(['CustomerID', 'ChurnReason', 'ChurnCategory', 'CustomerStatus', 'City'])
cat_cols = cat_cols.drop(['Country', 'State', 'Quarter'])
cat_cols = cat_cols.drop(['Under30', 'SeniorCitizen']).tolist()

cat_cols = [col for col in cat_cols if col != TARGET]

for col in cat_cols:

    print("\n" + "=" * 80)
    print(col)
    print("=" * 80)

    display(
        pd.crosstab(
            df[col],
            df[TARGET],
            dropna=False,
            normalize="index"
        )
        .mul(100)
        .round(2)
    )

### 3.4 Data readiness

Síntese da EDA: o dado está pronto para modelar? Quais tratamentos serão necessários?

### 3.4 Data readiness

**Veredito:** dataset pronto para modelagem — sem bloqueios de qualidade.
Os tratamentos necessários estão definidos abaixo e serão aplicados no pipeline (Seção 5), nunca no `df` cru.

#### Integridade
- 7.043 linhas × 50 colunas. Sem duplicatas (`CustomerID` 100% único).
- Sem valores numéricos impossíveis (todos `min ≥ 0`, máximos plausíveis).
- Nulos concentrados em 4 colunas, todos com tratamento definido (abaixo).

#### Alvo
- `ChurnLabel`: 26,5% de churn (1.869 / 5.174) → desbalanceamento com ratio de 0,36.
- Implica: split **estratificado**, métrica sensível a desbalanceamento (Seção 4) e avaliar `class_weight="balanced"`.

#### Features removidas (detalhe em docs/Notas_para_model_card.md)
- **Identificador:** CustomerID
- **Leakage:** CustomerStatus, ChurnScore, ChurnCategory, ChurnReason
- **Precaução:** CLTV
- **Redundância:** ZipCode, SeniorCitizen, Under30, TotalRevenue, InternetService
- **Constantes (variância zero):** Country, State, Quarter
- **Alta cardinalidade / geo redundante:** City

#### Decisão pendente
- `SatisfactionScore`: leakage suspeito (doc sem âncora temporal + separação 100/0%). Será testada **com e sem** no baseline; decisão final após comparar.

#### Tratamentos para o pipeline (Seção 5)
| Tratamento | Colunas |
|---|---|
| Imputar valor de negócio | Offer → "No offer"; InternetType → "No internet" |
| One-hot encoding | categóricas (binárias + Contract, PaymentMethod, Offer, InternetType) |
| StandardScaler | numéricas em geral |
| Testar log1p / RobustScaler | TotalLongDistanceCharges, AvgMonthlyGBDownload (cauda direita, skew ~1,2) |
| Remoção de linhas | nenhuma (outliers verificados como legítimos) |
| Mapear alvo | ChurnLabel Yes/No → 1/0 |
  
> Todo o pré-processamento vai **dentro do Pipeline**, ajustado só no treino, para não vazar a distribuição do teste.

#### Sinais observados (insumo para a modelagem)
- `TenureinMonths` é o preditor numérico mais forte (correlação negativa de -0.352861, indicando que cliente tem churn maior).
- `Married e Dependents` possuem elevada capacidade preditora, indicando que ter mais pessoas usando tende a reduzir o churn.
- `ReferredaFriend` é bom preditor, significando que indicações indicam satisfação e possível permanência
- `Offer` discrimina: "No offer" ~27% vs Offer E 53%.
- `Contract, PaperlessBilling e PaymentMethod` tem elevado poder discriminatório.
- `PhoneService e MultipleLines` tem baixo poder discriminatório, indicando que serviço de telefonia não é diferencial.


## 4. Definição de métricas

#### **Métricas técnicas**
`Acurácia ((TP+TN)/total)` — Taxa de acerto geral. Ruim sob desbalanceamento.  
`Precisão (TP/(TP+FP))` — A taxa de acerto das previsões positivas.  
`Recall (TP/(TP+FN))` — A taxa de acerto dos casos realmente positivos.  
`F1` — Média harmônica de P e R.  
`ROC-AUC` — Área sob a curva TPR × FPR (FPR = FP/(FP+TN)).  
`PR-AUC (principal)` — Área sob a curva Precisão × Recall.  


#### **Métricas de negócio**
`custo = c_FN · (falsos negativos) + c_FP · (falsos positivos)`  
  
**Onde:**  
**c_FN** é o custo da perda de clientes que o modelo previu permanência incorretamente  
**c_FP** é o custo das campanhas e/ou tratamentos para clientes que o modelo previu churn incorretamente  
  

`Os pesos c_FN e c_FP serão definidos quando a estratégia de retenção for conhecida.`

## 5. Pré-processamento e split

Split estratificado treino/teste com seed fixa.

In [ ]:
# Cópia do dataset original e imputação de valores

df_model = df.copy()

# Imputação de valores para Offer e InternetType
df_model["Offer"]        = df_model["Offer"].fillna("No offer")
df_model["InternetType"] = df_model["InternetType"].fillna("No internet")

# Criação de Y, com valores 0/1
y = df_model["ChurnLabel"].map({"No": 0, "Yes": 1})


In [ ]:
# Identifição de grupos de colunas e cenários

COLS_DROP_BASE = [
    "ChurnLabel",                                                              # vira y
    "CustomerID",                                                              # id
    "CustomerStatus", "ChurnScore", "ChurnCategory", "ChurnReason",            # leakage
    "CLTV",                                                                    # precaução
    "ZipCode", "SeniorCitizen", "Under30", "TotalRevenue", "InternetService",  # redundância
    "Country", "State", "Quarter",                                             # constantes
    "City",                                                                    # alta cardinalidade
    #"SatisfactionScore",                                                       # leakage suspeito → cenário "SEM"
]

SKEWED = ["TotalLongDistanceCharges", "AvgMonthlyGBDownload"]

cenarios = {
    "sem_satisfaction": COLS_DROP_BASE + ["SatisfactionScore"],
    "com_satisfaction": COLS_DROP_BASE,
}


In [ ]:
# Definição do experimento e métricas de performance

mlflow.set_experiment("customer_churn_prediction")

resultados = {}

def logar_metricas(y_train, y_pred_train, y_test, y_pred_test, y_proba_test):
    """Calcula e loga o MESMO conjunto de métricas para qualquer modelo,
    garantindo que todos os runs do MLflow sejam comparáveis."""
    m = {
        "train_accuracy": accuracy_score(y_train, y_pred_train),
        "test_accuracy":  accuracy_score(y_test,  y_pred_test),
        "test_f1_score":  f1_score(y_test, y_pred_test),
        "test_precision": precision_score(y_test, y_pred_test),
        "test_recall":    recall_score(y_test, y_pred_test),
        "test_roc_auc":   roc_auc_score(y_test, y_proba_test),
        "test_pr_auc":    average_precision_score(y_test, y_proba_test),
    }
    m["overfitting"] = m["train_accuracy"] - m["test_accuracy"]
    for nome, valor in m.items():
        mlflow.log_metric(nome, valor)
    return m


2026/06/24 22:08:24 INFO mlflow.tracking.fluent: Experiment with name 'customer_churn_prediction' does not exist. Creating a new experiment.


## 6. Baselines, seleção de métricas e Model Registry

Nesta seção treinamos dois baselines — DummyClassifier (piso de referência) e Regressão Logística — e registramos parâmetros, métricas e artefatos no MLflow. A comparação é feita com e sem SatisfactionScore para quantificar a suspeita de data leakage levantada na EDA. O versionamento do dataset será incorporado no próximo notebook (MLP).

`DummyClassifier` (referência trivial)

In [ ]:
# Modelo com DummyClassifier, para ser usado como comparação

y_train, y_test = train_test_split(y, test_size=0.2, stratify=y, random_state=SEED)

X_ph_train = np.zeros((len(y_train), 1))   # placeholder: stratified ignora as features
X_ph_test  = np.zeros((len(y_test),  1))

with mlflow.start_run(run_name="dummy_stratified"):
    mlflow.log_param("modelo", "dummy_stratified")

    dummy = DummyClassifier(strategy="stratified", random_state=SEED)
    dummy.fit(X_ph_train, y_train)

    metr = logar_metricas(
        y_train, dummy.predict(X_ph_train),
        y_test,  dummy.predict(X_ph_test),
        dummy.predict_proba(X_ph_test)[:, 1],
    )
    mlflow.sklearn.log_model(dummy, "model")

    resultados["dummy"] = metr

    print(f"=== DUMMY ===")
    for k, v in metr.items():
        print(f"{k:<16}{v:.4f}")
    print()



`Regressão Logística` (Avaliando se SatisfactionScore é data leakage)

In [ ]:
# Dois cenários: Com e sem SatisfactionScore

for nome, cols_drop in cenarios.items():
    X = df_model.drop(columns=cols_drop)
    X[SKEWED] = np.log1p(X[SKEWED])
    X = pd.get_dummies(X, columns=X.select_dtypes("object").columns, drop_first=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED
    )
    
    num_cols  = [c for c in X_train.select_dtypes("number").columns]

    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])


    with mlflow.start_run(run_name=f"logreg_{nome}"):
        mlflow.log_param("modelo", "logistic_regression")
        mlflow.log_param("inclui_satisfaction", "com" in nome)
        mlflow.log_param("n_features", X_train.shape[1])

        model = LogisticRegression(random_state=SEED, max_iter=1000)
        model.fit(X_train, y_train)

        metr = logar_metricas(
            y_train, model.predict(X_train),
            y_test,  model.predict(X_test),
            model.predict_proba(X_test)[:, 1],
        )
        mlflow.sklearn.log_model(model, "model")
        
        resultados[f"logreg_{nome}"] = metr

        print(f"=== LOGREG [{nome}] ===")
        for k, v in metr.items():
            print(f"{k:<16}{v:.4f}")
        print()


`Análise de resultados`

In [8]:
tabela = pd.DataFrame(resultados).round(4)
display(tabela)

,dummy,logreg_sem_satisfaction,logreg_com_satisfaction
train_accuracy,0.6207,0.8468,0.9625
test_accuracy,0.6217,0.8524,0.9645
test_f1_score,0.2903,0.7054,0.9307
test_precision,0.2891,0.7500,0.9655
test_recall,0.2914,0.6658,0.8984
test_roc_auc,0.5163,0.9108,0.9920
test_pr_auc,0.2723,0.7857,0.9819
overfitting,-0.0010,-0.0056,-0.0020


In [ ]:
runs = mlflow.search_runs(experiment_names=["customer_churn_prediction"])
runs[["tags.mlflow.runName", "metrics.test_pr_auc", "metrics.test_recall", "metrics.test_f1_score"]]

,tags.mlflow.runName,metrics.test_pr_auc,metrics.test_recall,metrics.test_f1_score
0,logreg_com_satisfaction,0.981946,0.898396,0.930748
1,logreg_sem_satisfaction,0.785696,0.665775,0.705382
2,dummy_stratified,0.272340,0.291444,0.290280


### Resumo dos baselines

O **DummyClassifier (stratified)** estabelece o piso de performance: PR-AUC de **0,272** (próximo à prevalência de churn de 26,5%), confirmando que qualquer modelo acima disso está aprendendo sinal real dos dados.

A **Regressão Logística** sem `SatisfactionScore` atinge PR-AUC de **0,786** — um lift substancial sobre o piso, obtido pela combinação de múltiplas features moderadas (nenhuma com correlação individual > 0,35). Não apresenta overfitting (diferença treino-teste de -0,006).

Ao incluir `SatisfactionScore`, a PR-AUC salta para **0,982** — um ganho desproporcional que confirma a suspeita levantada na EDA: a variável é **data leakage**. Esse score de satisfação é coletado após o evento de churn (ou é derivado dele), tornando-o indisponível no momento da predição em produção.

**Decisão: `SatisfactionScore` será excluída definitivamente do estudo.** O modelo honesto (sem ela) já demonstra capacidade preditiva forte, e mantê-la inflaria artificialmente as métricas sem refletir performance real em deploy.

## 7. Conclusões e próximos passos


**Conclusões**  
A EDA revelou um dataset de 7.043 clientes com 26,5% de churn (desbalanceamento moderado, ratio 0,36), sem bloqueios de qualidade. Das 50 colunas originais, 16 foram removidas por leakage, redundância, constância ou alta cardinalidade. SatisfactionScore, inicialmente classificada como leakage suspeito, foi confirmada como data leakage — a PR-AUC da Regressão Logística salta de 0,786 para 0,982 ao incluí-la, um ganho incompatível com uma feature legítima. Será excluída definitivamente.

O DummyClassifier (stratified) estabeleceu o piso em PR-AUC 0,272 (≈ prevalência). A Regressão Logística sem SatisfactionScore alcançou PR-AUC de 0,786 sem overfitting, demonstrando que o conjunto de features moderadas (nenhuma com correlação > 0,35) carrega sinal preditivo forte quando combinadas.

**Próximos passos**  
Treinar MLP em PyTorch e comparar com os baselines (≥ 4 métricas)
Testar class_weight="balanced" na Regressão Logística para avaliar trade-off de recall (atual: 0,666 — perde 1/3 dos churners)
Analisar trade-off de custo falso positivo × falso negativo
Versionar o dataset no MLflow
Refatorar pré-processamento para Pipeline/ColumnTransformer (Marco 3)
Elaborar ML Canvas e Model Card